# MARL-BattleGrounds Quick Start

Play games, evaluate a trained team and read the results on a free notebook GPU.

[Open in Colab](https://colab.research.google.com/github/oceansystemslab/MARL-BattleGrounds/blob/v1.0.0/examples/quickstart.ipynb) · [Open in Kaggle](https://kaggle.com/kernels/welcome?src=https://github.com/oceansystemslab/MARL-BattleGrounds/blob/v1.0.0/examples/quickstart.ipynb)

- On Colab, pick a GPU first: Runtime > Change runtime type > T4 GPU.
- On Kaggle, turn on Internet and pick the T4 GPU in the notebook's settings.
- If Colab asks you to restart the session after the first cell, restart it and run all cells again. The first cell then finishes at once.

The BattleClient and the Replay Viewer, where you play and watch games, do not run inside a notebook service: their pages only accept `127.0.0.1`. Install MARL-BattleGrounds on your own computer for them: https://marl-battlegrounds.com/#start.

In [ ]:
# On Colab or Kaggle, install MARL-BattleGrounds with the JAX build that
# matches this notebook's accelerator. Anywhere else this cell does nothing;
# install first with https://marl-battlegrounds.com/#start.
import importlib.metadata
import os
import subprocess
import sys

if "google.colab" in sys.modules or "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    installed = {
        (distribution.metadata["Name"] or "").lower()
        for distribution in importlib.metadata.distributions()
    }
    builds = (("jax-cuda13-plugin", "cuda13"), ("jax-cuda12-plugin", "cuda12"))
    builds += (("libtpu", "tpu"),)
    build = next((name for plugin, name in builds if plugin in installed), None)
    extras = "all" if build is None else f"all,{build}"
    package = f"marl-battlegrounds[{extras}]==1.0.0"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)
    print(f"Installed {package}.")

## Play 128 Games At Once

`make` gives a batch of games. `reset` and `step` work like other JAX environments, and `reset_done` restarts only the games that ended. Here every agent takes random legal actions.

In [ ]:
import jax

import marl_battlegrounds as marl_bgs

print("JAX runs on:", jax.devices())
env = marl_bgs.make("tdm", map_id=0, num_envs=128)
key = jax.random.key(0)
key, reset_key = jax.random.split(key)
obs, state = env.reset(reset_key)
for _ in range(100):
    key, action_key, step_key, reset_key = jax.random.split(key, 4)
    actions = env.sample_actions(action_key, state)
    obs, state, reward, done, info = env.step(step_key, state, actions)
    obs, state = env.reset_done(reset_key, state)
jax.block_until_ready(state)
print("Played 100 steps in each of 128 games.")

## Evaluate A Trained Team

Download the trained Recurrent MAPPO team from Hugging Face and play 32 games against the built-in TDM-Alpha team on the five test maps (the default). The trained team is Team A in every game; paired games swap the two spawn ends.

In [ ]:
trained = marl_bgs.load_method("MARL-BattleGrounds/tdm-season-0-recurrent-mappo")
result = marl_bgs.evaluate(trained, "tdm-alpha", num_episodes=32, seed=0, num_envs=32)
print("Games played:", len(result.completed_episode_ids))

## Read The Results

Your team's games against the opponent: wins, draws and losses, and mean points for and against. The point margin is points for minus points against. The second table splits the same games by map. (`team_a_rows` keeps your team's rows; the full table also holds the opponent's view of the same games.)

In [ ]:
import pandas as pd
from IPython.display import display


def team_a_rows(games: marl_bgs.EvaluationResult, by: str = "overall") -> pd.DataFrame:
    """Return the head-to-head rows seen from Team A, your team.

    The full table also holds the opponent's view of the same games.
    by is "overall" (one row) or "map" (one row per map).
    """
    team_a = games.table("episodes")["team_a_policy"][0]
    table = pd.DataFrame(games.head_to_head(by=by))
    return table[table["system_name"] == team_a]


display(team_a_rows(result))
display(team_a_rows(result, by="map"))

## Play The Season 0 Field

Play your team against the nine Season 0 teams: the six trained teams and the three built-in TDM bots, one pair of games on each test map. This is a quick local check, not a leaderboard result: official leaderboard runs are done by the maintainers from a submission, under the [tournament rules](https://github.com/oceansystemslab/MARL-BattleGrounds/blob/v1.0.0/docs/tournament_rules.md). Replace `trained` with your own team.

In [ ]:
# Your team plays every Season 0 team: one pair of games on each test map.
season_0 = {
    "Recurrent IPPO": "MARL-BattleGrounds/tdm-season-0-recurrent-ippo",
    "Recurrent MAPPO": "MARL-BattleGrounds/tdm-season-0-recurrent-mappo",
    "Feedforward IPPO": "MARL-BattleGrounds/tdm-season-0-feedforward-ippo",
    "Feedforward MAPPO": "MARL-BattleGrounds/tdm-season-0-feedforward-mappo",
    "PQN-VDN": "MARL-BattleGrounds/tdm-season-0-pqn-vdn",
    "QMIX": "MARL-BattleGrounds/tdm-season-0-qmix",
    "TDM-Alpha": "tdm-alpha",
    "TDM-Beta": "tdm-beta",
    "TDM-Gamma": "tdm-gamma",
}
your_team = trained  # or marl_bgs.load_method("<your saved model folder>")
rows = []
for name, opponent in season_0.items():
    games = marl_bgs.evaluate(your_team, opponent, num_episodes=10, seed=0, num_envs=32)
    row = team_a_rows(games).iloc[0]
    columns = ["games", "wins", "draws", "losses", "mean_point_margin"]
    rows.append({"Opponent": name} | row[columns].to_dict())
display(pd.DataFrame(rows))

## Next

- Explore saved results with every metric: [the analysis notebook](https://colab.research.google.com/github/oceansystemslab/MARL-BattleGrounds/blob/v1.0.0/examples/evaluation_analysis.ipynb).
- Install on your own machine to play in the BattleClient and watch replays: https://marl-battlegrounds.com/#start.
- Train your own team: https://github.com/oceansystemslab/MARL-BattleGrounds/blob/v1.0.0/docs/training.md.